In [ ]:
import sys, pygame, math, random, pygame.gfxdraw, pygame.transform

In [ ]:
from ball import *

In [ ]:
def get_distance(x1,x2,y1,y2):
    distance = -1
    dx = x2 - x1
    dy = y2 - y1
    distance = pow(dx,2)+pow(dy,2)
    distance = math.sqrt(distance)
    return distance

In [ ]:
def magnitude(vector):
    partials = []
    for dim in vector:
        partial = dim**2
        partials.append(partial)
    return math.sqrt(sum(partials))

In [ ]:
def two_balls_touching(b1,b2):
    #if the two balls are touching
    radius1 = (b1.rect.size[1] * b1.size)/2
    radius2 = (b2.rect.size[1] * b2.size)/2
    distance = get_distance(b1.get_x(),b2.get_x(),b1.get_y(),b2.get_y())
    if distance <= (radius1 + radius2):#the distance between them is less than both their radii
        return True
    else: return False

In [ ]:
def point_from_angle(origin, angle, radius):
    x = (math.cos(math.radians(angle))*radius) + origin[0]
    y = (math.sin(math.radians(angle))*radius) + origin[1]
    point = [x,y]
    return point

In [ ]:
def optimal_ball_location(origin, radius, nodes): 
    #find the next point by simple gradient descent
    degrees = 360
    point = [10,10] #dummy
    
    xpoints = []
    ypoints = []
    for i in range(degrees):
        xpoints.append(i)
        
    for degree in xpoints:
        #print(degree)
        #find the point on the circle from which the distance will be calculated
        x = (math.cos(math.radians(degree))*radius) + origin[0]
        y = (math.sin(math.radians(degree))*radius) + origin[1]
        point = [x,y]
        #find the sum of distance from all 'nodes' to this point
        distance = 0
        for node in nodes:
            partialdistance = get_distance(x,node.get_x(),y,node.get_y())
            if partialdistance == 0:
                partialdistance = 0.0000001
            distance += 1/partialdistance
        #print(point)
        #print(distance)
        ypoints.append(distance)
    
    #choose a random degree value (0-360)
    pointerx = random.randint(1,degrees)
    margin = 70 #to start
    while margin > 0:
        # find the slope from neighboring points (start with a margin of 25 each side)
        leftx = int(pointerx-margin)
        if leftx < 0:
            leftx += degrees
        rightx = int(pointerx+margin)
        if rightx >= 360:
            rightx -= degrees
        slope = ((ypoints[rightx]-ypoints[leftx])/(margin*2))#rise/run
        #print(pointerx)
        #print(slope)
        # move in the direction of neighboring slope (same margin)
        if slope >= 0:
            pointerx -= margin
        else:
            pointerx += margin
        if pointerx >= degrees:
            pointerx -= degrees
        if pointerx < 0:
            pointerx += degrees
        # do this until the margin is 0
        margin -= 5
    #print('radius',radius)
    #print(pointerx)
    point = point_from_angle(origin,pointerx,radius)
    return [point,pointerx]

In [ ]:
def prescribe_ball_location(origin,radius,previous):
    #the point is always the golden angle away from the last one.
    point = [10,10] #dummy
    newangle = previous + 138
    if newangle > 360:
        newangle -= 360
    point = point_from_angle(origin,newangle,radius)
    #print(newangle) #THIS IS USEFUL
    return [point,newangle]

In [ ]:
def new_ball(ball_list,playerball,previousangle,dt):
    start_speed = 20 * dt #FOR SOME REASON DT MAKES THE NEW BALLS MOVE FASTER!!!
    print(start_speed,"start speed")
    #Place a new ball away from balls in a certain radius...
    #?combine vectors, normalize, and reverse?? 
    #or do a gradient descent algorithm...
    if playerball.cooldown <= 0:
        playerball.cooldown = 1 #reset cooldown
        playerball.resize(playerball.size*0.995)
        #create the new seed primordium
        newball = Ball(playerball.get_x(),playerball.get_y(),0,0)
        newball.maxsize = 0.3
        if playerball.B and not playerball.C:
            newball.set_image("yellow_p_centered.png")
            newball.maxsize = 0.5
            newball.resize(newball.maxsize)
        if playerball.B and playerball.C:
            newball.set_image("yellow_seed.png")
            newball.maxsize = 0.3
            newball.resize(newball.maxsize)
        newball.resize(0.01)
        
        #find the list of local balls to the playerball
        local_dist = 700
        local_balls = []
        for ball in ball_list:
            distance = get_distance(playerball.get_x_center(),ball.get_x_center(),playerball.get_y_center(),ball.get_y_center())
            if distance < local_dist:
                local_balls.append(ball)
        #print(len(local_balls))
        
        #find an optimal location for the new ball
        playerballradius = playerball.get_radius_inner()
        playerballcenter = playerball.get_center()
        
        answer = prescribe_ball_location([playerballcenter[0],playerballcenter[1]],playerballradius,previousangle)
        #answer = optimal_ball_location(playerballcenter,playerballradius,local_balls)
        newball_location = answer[0]
        #find the angle of the ball coming out and find the deviation from last ball.
        previousangle = answer[1]
        print(previousangle)
        
        newball.set_center(newball_location[0],newball_location[1])
        #TODO: set the newball moving directly away from playerball
        #find the slope between the center of the playerball and the center of the newball
        ydev = playerballcenter[1]-newball.get_center()[1]
        xdev = playerballcenter[0]-newball.get_center()[0]
        #set the newball moving along a product of that slope
        newball.dx = -xdev * start_speed
        newball.dy = -ydev * start_speed
        
        #TODO: Rotate the petal to be facing directly away from playerball.
            #find the vector from pball to petal. 
            #convert the vector to the angle.
            #rotate the petal along this angle.
        ncenter = newball.get_center()
        pcenter = playerball.get_center()
        vector = [(ncenter[0]-pcenter[0]),-(ncenter[1]-pcenter[1])]
        if vector[0] == 0:
            vector[0] = 0.001
        angle = math.degrees(math.atan(vector[1]/vector[0]))
        if vector[0]<= 0:
            angle +=180
        #if angle < 0:
            #angle += 360 #NO DIFFERENCE
        newball.rotate_point(ncenter,angle-90) #ALWAYS FACES IN IF ON LEFT AND OUT IF ON RIGHT.
        #COULD BE RELATED TO: 
        #X VALUE (negative vector[0])
        #or 
        
        ball_list.append(newball)
        
    return ball_list, previousangle

In [ ]:
pygame.init()
t = 0
dt = 0.001

clock = pygame.time.Clock()
size = width, height = 700, 700
center_of_screen = (width//2,height//2)
black = 0, 0, 0
light_blue = 120,120,250
blue = 90,90,210
bg_color = light_blue

screen = pygame.display.set_mode(size)

speedmod = 0.9965 #998 is a good one

stem_img = pygame.image.load("stem.png")
stem_rect = stem_img.get_rect()
stem_rect.centerx = 350
stem_rect.y = 350

ball_list = []

#create the meristem.
playerball = Ball(100,100,0,0) #the x and y values are not perm.
playerball.set_image("meristem.png")
playerball.resize(0.6)
#move the ball to the center
playerball.set_center(center_of_screen[0],center_of_screen[1])
#playerball.set_x(100)

playerball.A = True
playerball.B = True
playerball.C = False

pressA = False
pressB = False
pressC = False

#create the back of the sunflower.
back = Ball(100,100,0,0)
back.set_image("brown_circle.png")
back.set_center(center_of_screen[0],center_of_screen[1])

playerball.cooldown = 2
previousangle = 0

back.set_radius_inner(playerball.get_radius_inner())

while True:
    for event in pygame.event.get():
        if event.type == pygame.QUIT: sys.exit()
    
    #move the balls
    for ball in ball_list:
        ball.move(0.017) #FOR SOME REASON DT MAKES THE BALLS MOVE FASTER
        #TODO TODO: they should not slow down at first, then slow down a lot.
        #exponential?
        ball.dx = ball.dx * speedmod 
        ball.dy = ball.dy * speedmod 
        
        ball_vector = ball.vector()
        ball_magnitude = magnitude(ball_vector)
        print("{0:0.2f}".format(round(ball_magnitude, 2)))
        #the speed lessens over time. 
        '''
        ball_magnitude *= 2
        ball.dx *= speedmod
        ball.dy *= speedmod
        '''
        
        
    #affect the player's movement: NOT NECESSARY
    keys = pygame.key.get_pressed()
    if keys[pygame.K_a]:
        #toggle hormone A
        if not pressA:
            pressA = True
            playerball.A = not playerball.A
    else:
        pressA = False
        
    if keys[pygame.K_b]:
        #toggle hormone B
        if not pressB:
            pressB = True
            playerball.B = not playerball.B
    else:
        pressB = False
        
    if keys[pygame.K_c]:
        #toggle hormone C
        if not pressC:
            pressC = True
            playerball.C = not playerball.C
    else:
        pressC = False

    #cooldown the meristem (playerball where new balls will shoot off)
    if playerball.cooldown > 0:
        playerball.cooldown -= dt

    print("dT:",dt)
    #create a new seed primordium, if the meristem is ready. 
    ball_list, previousangle = new_ball(ball_list, playerball, previousangle,0.017) #TODO!!! DT is wrong
    
    #each element of the plant will grow over time!
    for ball in ball_list:
        ball_size = ball.size
        if ball_size < ball.maxsize:
            """
            a = (-0.1*dt)+1
            ball_size = ball.maxsize*(ball_size**(a))
            """
            ball_size += 0.3 * dt
            if ball_size > ball.maxsize:
                ball_size = ball.maxsize
            ball.resize(ball_size)

    # make the size of the back of the sunflower reach to the EDGE of the farthest newball (or the petals.)
        # After the hormonal changes are implemented
    # find the coords of the furthest (first) seed.
    far_center = [0,0]
    ball_radius = 0
    if len(ball_list) >= 1:
        far_center = ball_list[0].get_center()
        ball_radius = ball_list[0].get_radius_inner()
        # find the distance of the furthest seed. 
        far_distance = get_distance(far_center[0],center_of_screen[0],far_center[1],center_of_screen[1])
        # find the distance of its edge
        #far_distance += ball_radius
        # (make it the new radius of back.)
        back.set_radius_inner(far_distance)
        # the default radius of back is 250.

    #these arrays contain all of the images and rects to be rendered, in order from bottom to top. 
    render_imgs = []
    render_rects = [] 
    #add these in the correct order
    render_imgs.append(stem_img)
    render_rects.append(stem_rect)
    #add the back
    render_imgs.append(back.image)
    render_rects.append(back.rect)
    for ball in ball_list:
        render_imgs.append(ball.image)
        render_rects.append(ball.rect)
    #add the meristem
    render_imgs.append(playerball.image)
    render_rects.append(playerball.rect)
    
    #RENDERING STAGE
    screen.fill(bg_color)
    #blit every part of the sunflower.
    for i in range(len(render_imgs)):
        screen.blit(render_imgs[i],render_rects[i])
    pygame.display.flip()

    # limits FPS to 60
    # dt is delta time in seconds since last frame, used for framerate-
    # independent physics.
    dt = clock.tick(60) / 1000
    t += dt